# 01 - Docker ve Kafka Ortam Kurulumu

Bu notebook projenin altyapı katmanını açıklar. Amaç, tüm büyük veri bileşenlerini Docker Compose ile izole ve tekrar üretilebilir bir ortamda çalıştırmaktır.

## Bu adımda kurulan servisler

| Servis | Görevi |
|---|---|
| Kafka | Streaming veri akışı için mesaj kuyruğu |
| Producer | MovieLens rating verilerini Kafka'ya gönderen Python uygulaması |
| Spark | Kafka'dan veri okuyup Delta Lake'e yazan işleme motoru |
| MLflow | Model deney takibi |
| Dashboard | Gold katmanı sonuçlarını görselleştiren Streamlit arayüzü |

## Bu adımın proje mimarisindeki yeri

Docker ortamı, pipeline'ın temel altyapısıdır. Diğer bütün adımlar bu container'lar üzerinde çalışır.

## docker-compose.yml
Aşağıdaki dosya tüm servisleri tek merkezden yönetir.

```bash
services:
  kafka:
    image: apache/kafka:3.7.0
    container_name: movie-kafka
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_LISTENERS: PLAINTEXT://kafka:29092,CONTROLLER://kafka:9093,PLAINTEXT_HOST://0.0.0.0:9092
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:29092,PLAINTEXT_HOST://localhost:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT,PLAINTEXT_HOST:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"

  mlflow:
    image: ghcr.io/mlflow/mlflow:v2.14.1
    container_name: movie-mlflow
    ports:
      - "5000:5000"
    volumes:
      - ./mlruns:/mlruns
    command: >
      mlflow server
      --backend-store-uri /mlruns
      --default-artifact-root /mlruns
      --host 0.0.0.0
      --port 5000

  producer:
    build:
      context: ./producer
    container_name: movie-producer
    depends_on:
      - kafka
    volumes:
      - ./data:/app/data
    environment:
      KAFKA_BOOTSTRAP_SERVERS: kafka:29092
      KAFKA_TOPIC: movie-ratings

  spark:
    build:
      context: ./spark
    container_name: movie-spark
    depends_on:
      - kafka
      - mlflow
    volumes:
      - ./data:/app/data
      - ./delta:/app/delta
      - ./mlruns:/app/mlruns
    environment:
      KAFKA_BOOTSTRAP_SERVERS: kafka:29092
      KAFKA_TOPIC: movie-ratings
      MLFLOW_TRACKING_URI: http://mlflow:5000

  dashboard:
    build:
      context: ./dashboard
    container_name: movie-dashboard
    depends_on:
      - spark
      - mlflow
    ports:
      - "8501:8501"
    volumes:
      - ./delta:/app/delta

```

## Producer Dockerfile
Producer servisi Python imajı üzerinde çalışır ve Kafka'ya veri gönderir.

```bash
FROM python:3.11-slim-bookworm

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY kafka_producer.py .

CMD ["python", "kafka_producer.py"]

```

## Producer requirements.txt

In [ ]:
kafka-python==2.0.2
pandas==2.2.2

## Spark Dockerfile
Spark container içinde PySpark, Delta Lake ve MLflow bağımlılıkları bulunur.

```bash
FROM python:3.11-slim-bookworm

WORKDIR /app

RUN apt-get update && apt-get install -y \
    openjdk-17-jdk \
    curl \
    procps \
    && rm -rf /var/lib/apt/lists/*

ENV JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
ENV PATH="$JAVA_HOME/bin:$PATH"

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY streaming_to_delta.py .
COPY train_model.py .
COPY check_bronze.py .
COPY bronze_to_silver.py .
COPY silver_to_gold.py .
COPY feature_engineering.py .
COPY train_regression_models.py .

CMD ["tail", "-f", "/dev/null"]

```

## Spark requirements.txt

In [ ]:
pyspark==3.5.1
delta-spark==3.2.0
mlflow==2.14.1
pandas==2.2.2

## Dashboard Dockerfile
Dashboard Streamlit ile çalışır ve 8501 portundan yayın yapar.

```bash
FROM python:3.11-slim-bookworm

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8501

CMD ["streamlit", "run", "app.py", "--server.address=0.0.0.0", "--server.port=8501"]

```

## Dashboard requirements.txt

In [ ]:
streamlit==1.37.1
pandas==2.2.2
pyarrow==16.1.0
plotly==5.22.0

## Çalıştırma komutları

Proje ana dizininde çalıştırılır.

```bash
docker compose build
docker compose up -d kafka mlflow spark dashboard
docker ps

```

## Beklenen çıktı

`docker ps` çıktısında aşağıdaki container'lar görülmelidir:

- movie-kafka
- movie-mlflow
- movie-spark
- movie-dashboard

Bu noktada altyapı hazırdır.